In [1]:
pip install xgboost

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: C:\Users\LENOVO\AppData\Local\Programs\Python\Python314\python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd

In [3]:
from sklearn.model_selection import train_test_split
from sklearn import metrics
from sklearn.metrics import accuracy_score

In [4]:
# Load your dataset
df = pd.read_csv("../data/BlackFriday.csv")

# Split into train and test (80% train, 20% test)
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

print("Train size:", len(train_df))
print("Test size:", len(test_df))

Train size: 440054
Test size: 110014


In [5]:
from sklearn.preprocessing import RobustScaler

# Separate numeric columns
numeric_cols = train_df.select_dtypes(include=['int64','float64']).columns

# Initialize scaler
scaler = RobustScaler()

# Fit on train numeric data
train_df[numeric_cols] = scaler.fit_transform(train_df[numeric_cols])

# Transform test numeric data
test_df[numeric_cols] = scaler.transform(test_df[numeric_cols])

In [6]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score
from xgboost import XGBRegressor

# Load dataset
df = pd.read_csv("../data/BlackFriday.csv")

# Drop IDs
df = df.drop(columns=['User_ID','Product_ID'])

# Split into train and test
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

# Features and target
X_train = train_df.drop(columns=['Purchase'])
y_train = train_df['Purchase']
X_test = test_df.drop(columns=['Purchase'])
y_test = test_df['Purchase']

# Categorical columns
categorical_cols = X_train.select_dtypes(include=['object']).columns

# Preprocessor: encode categorical, pass through numeric untouched
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ],
    remainder='passthrough'
)

# Pipeline: preprocessing + XGBoost
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', XGBRegressor(
        n_estimators=400,       # number of trees
        learning_rate=0.2,      # step size shrinkage
        max_depth=9,            # tree depth
        subsample=0.8,          # row sampling
        colsample_bytree=0.8,   # feature sampling
        random_state=42,
        n_jobs=-1
    ))
])

# Fit model
model.fit(X_train, y_train)

# Predict
y_pred = model.predict(X_test)

# Evaluate
print("MSE:", mean_squared_error(y_test, y_pred))
print("R² Score:", r2_score(y_test, y_pred))

MSE: 8412958.0
R² Score: 0.6651716232299805


In [7]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score
from xgboost import XGBRegressor

# Load dataset
df = pd.read_csv("../data/BlackFriday.csv")

# Drop IDs
df = df.drop(columns=['User_ID','Product_ID'])

# Split into train and test
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

# Features and target
X_train = train_df.drop(columns=['Purchase'])
y_train = train_df['Purchase']
X_test = test_df.drop(columns=['Purchase'])
y_test = test_df['Purchase']

# Categorical columns
categorical_cols = X_train.select_dtypes(include=['object']).columns

# Preprocessor: encode categorical, pass through numeric untouched
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ],
    remainder='passthrough'
)

# Pipeline: preprocessing + XGBoost
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', XGBRegressor(
        n_estimators=400,       # number of trees
        learning_rate=0.1,      # step size shrinkage
        max_depth=8,            # tree depth
        subsample=0.8,          # row sampling
        colsample_bytree=0.8,   # feature sampling
        random_state=42,
        n_jobs=-1
    ))
])

# Fit model
model.fit(X_train, y_train)

# Predict
y_pred = model.predict(X_test)

# Evaluate
print("MSE:", mean_squared_error(y_test, y_pred))
print("R² Score:", r2_score(y_test, y_pred))

MSE: 8210551.0
R² Score: 0.6732271909713745


In [8]:
pip install catboost

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: C:\Users\LENOVO\AppData\Local\Programs\Python\Python314\python.exe -m pip install --upgrade pip


In [9]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from catboost import CatBoostRegressor

# Load dataset
df = pd.read_csv("../data/BlackFriday.csv")

# Drop IDs
df = df.drop(columns=['User_ID','Product_ID'])

# Split into train and test
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

# Features and target
X_train = train_df.drop(columns=['Purchase'])
y_train = train_df['Purchase']
X_test = test_df.drop(columns=['Purchase'])
y_test = test_df['Purchase']

# Identify categorical columns (CatBoost can handle them directly)
categorical_cols = X_train.select_dtypes(include=['object']).columns.tolist()

# Initialize CatBoost Regressor
model = CatBoostRegressor(
    iterations=500,        # number of boosting rounds
    learning_rate=0.1,     # step size
    depth=8,               # tree depth
    loss_function='RMSE',  # regression loss
    cat_features=categorical_cols,
    random_state=42,
    verbose=100            # print training progress every 100 iterations
)

# Fit model
model.fit(X_train, y_train)

# Predict
y_pred = model.predict(X_test)

# Evaluate
print("MSE:", mean_squared_error(y_test, y_pred))
print("R² Score:", r2_score(y_test, y_pred))

0:	learn: 4726.1982044	total: 1.15s	remaining: 9m 32s
100:	learn: 2925.9184621	total: 29.9s	remaining: 1m 58s
200:	learn: 2885.6730752	total: 43.9s	remaining: 1m 5s
300:	learn: 2868.7051904	total: 57.6s	remaining: 38.1s
400:	learn: 2854.9682738	total: 1m 11s	remaining: 17.7s
499:	learn: 2846.5104202	total: 1m 25s	remaining: 0us
MSE: 8338771.261890858
R² Score: 0.6681241923694774
